# T39 - 成交数据
## 第一章：环境配置

本章介绍成交数据采集系统的环境配置，包括：
1. 依赖包安装
2. 环境变量配置
3. 数据库连接测试
4. API连接测试

## 1. 依赖包安装

In [ ]:
# 安装必要的依赖包
# !pip install pandas sqlalchemy pymysql psycopg2-binary requests python-dotenv

## 2. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
from time import sleep
import logging

# 数据库连接
import sqlalchemy
from sqlalchemy.sql import text
import pymysql

# HTTP请求
import requests

# 环境变量
import os
from dotenv import load_dotenv

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('T39-成交数据')

print('依赖包导入成功！')

## 3. 环境变量配置

敏感信息（如密码）应通过环境变量配置，而非硬编码在代码中。

### 环境变量说明

| 变量名 | 说明 | 示例 |
|--------|------|------|
| MYSQL_HOST | MySQL主机地址 | rm-xxx.mysql.rds.aliyuncs.com |
| MYSQL_PORT | MySQL端口 | 3306 |
| MYSQL_USER | MySQL用户名 | hz_work |
| MYSQL_PASSWORD | MySQL密码 | your_password |
| RATINGDOG_USERNAME | 评级狗用户名 | your_phone |
| RATINGDOG_PASSWORD | 评级狗密码 | your_password |

In [ ]:
# 加载.env文件中的环境变量（如果存在）
load_dotenv()

# 显示当前环境变量配置状态
env_vars = [
    'MYSQL_HOST',
    'MYSQL_PORT', 
    'MYSQL_USER',
    'MYSQL_PASSWORD',
    'RATINGDOG_USERNAME',
    'RATINGDOG_PASSWORD'
]

print('=== 环境变量配置状态 ===')
for var in env_vars:
    value = os.getenv(var)
    if value:
        # 隐藏密码类型变量的值
        if 'PASSWORD' in var:
            print(f'{var}: ****** (已配置)')
        else:
            print(f'{var}: {value}')
    else:
        print(f'{var}: 未配置')

## 4. 加载配置模块

In [ ]:
# 导入配置模块
from config import Config, config

# 显示配置信息
print('=== 配置信息 ===')
print(f'MySQL主机: {config.database.mysql_host}')
print(f'MySQL端口: {config.database.mysql_port}')
print(f'MySQL用户: {config.database.mysql_user}')
print(f'MySQL YQ数据库: {config.database.mysql_database_yq}')
print(f'MySQL Bond数据库: {config.database.mysql_database_bond}')
print(f'评级狗API超时: {config.ratingdog_api.timeout}秒')
print(f'每页最大结果数: {config.ratingdog_api.max_result_count}')
print(f'最大重试次数: {config.ratingdog_api.max_retries}')

## 5. 数据库连接测试

In [ ]:
def test_mysql_connection(database='yq'):
    """测试MySQL数据库连接"""
    try:
        if database == 'yq':
            engine = sqlalchemy.create_engine(
                config.database.get_mysql_yq_connection_string(),
                poolclass=sqlalchemy.pool.NullPool
            )
        else:
            engine = sqlalchemy.create_engine(
                config.database.get_mysql_bond_connection_string(),
                poolclass=sqlalchemy.pool.NullPool
            )
        
        conn = engine.connect()
        
        # 执行测试查询
        result = conn.execute(text('SELECT 1'))
        result.fetchone()
        
        conn.close()
        return True, '连接成功'
    except Exception as e:
        return False, str(e)

# 测试YQ数据库连接
success, message = test_mysql_connection('yq')
print(f'YQ数据库连接: {"成功" if success else "失败"} - {message}')

# 测试Bond数据库连接
success, message = test_mysql_connection('bond')
print(f'Bond数据库连接: {"成功" if success else "失败"} - {message}')

## 6. 检查数据库表结构

In [ ]:
def check_table_structure():
    """检查相关表结构"""
    try:
        engine = sqlalchemy.create_engine(
            config.database.get_mysql_yq_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        conn = engine.connect()
        
        # 检查cjqb表是否存在
        try:
            result = conn.execute(text('DESCRIBE yq.cjqb'))
            columns = result.fetchall()
            print('=== yq.cjqb 表结构 ===')
            for col in columns:
                print(f'  {col[0]}: {col[1]}')
        except Exception as e:
            print(f'yq.cjqb 表不存在或无法访问: {e}')
        
        conn.close()
    except Exception as e:
        print(f'检查表结构失败: {e}')

check_table_structure()

In [ ]:
def check_source_table():
    """检查数据源表"""
    try:
        engine = sqlalchemy.create_engine(
            config.database.get_mysql_bond_connection_string(),
            poolclass=sqlalchemy.pool.NullPool
        )
        conn = engine.connect()
        
        # 检查marketinfo_abs表
        try:
            result = conn.execute(text('''
                SELECT COUNT(DISTINCT dt) as date_count 
                FROM bond.marketinfo_abs
            '''))
            row = result.fetchone()
            print(f'bond.marketinfo_abs 表日期数: {row[0]}')
            
            # 获取日期范围
            result = conn.execute(text('''
                SELECT MIN(dt), MAX(dt) 
                FROM bond.marketinfo_abs
            '''))
            row = result.fetchone()
            print(f'日期范围: {row[0]} ~ {row[1]}')
            
        except Exception as e:
            print(f'查询 bond.marketinfo_abs 表失败: {e}')
        
        conn.close()
    except Exception as e:
        print(f'检查源表失败: {e}')

check_source_table()

## 7. 评级狗API连接测试

In [ ]:
def test_ratingdog_login():
    """测试评级狗API登录"""
    try:
        response = requests.post(
            config.ratingdog_api.login_url,
            headers=config.ratingdog_api.get_login_headers(),
            json=config.ratingdog_api.get_login_data(),
            timeout=config.ratingdog_api.timeout
        )
        
        result = response.json()
        
        if result.get('success'):
            access_token = result['result']['accessToken']
            # 只显示token的前10个字符
            token_preview = access_token[:10] + '...' if len(access_token) > 10 else access_token
            return True, f'登录成功，Token: {token_preview}'
        else:
            error_msg = result.get('error', {}).get('message', '未知错误')
            return False, f'登录失败: {error_msg}'
            
    except Exception as e:
        return False, f'连接异常: {str(e)}'

# 测试登录
success, message = test_ratingdog_login()
print(f'评级狗API登录: {"成功" if success else "失败"} - {message}')

## 8. 环境配置总结

In [ ]:
print('='*60)
print('T39-成交数据 环境配置检查总结')
print('='*60)

checks = [
    ('MySQL YQ数据库', test_mysql_connection('yq')[0]),
    ('MySQL Bond数据库', test_mysql_connection('bond')[0]),
    ('评级狗API登录', test_ratingdog_login()[0]),
]

all_passed = True
for name, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {name}')
    if not passed:
        all_passed = False

print('='*60)
if all_passed:
    print('所有检查通过，环境配置完成！')
else:
    print('部分检查未通过，请检查配置后重试。')
print('='*60)